# 02 — Feature Engineering
Building features for the M5 demand forecasting models

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.feature_engine import (
    load_config, build_features,
    add_lag_features, add_rolling_features,
    add_calendar_features, add_fourier_features,
    add_price_features, add_encoding_features
)

config = load_config('../config/config.yaml')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_parquet('../data/processed/sales_merged.parquet')
print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Unique series: {df["id"].nunique()}')
df.head()

## 2. Demonstrate Each Feature Group
Working with a single item-store pair for clarity

In [ ]:
sample_id = df['id'].unique()[0]
sample = df[df['id'] == sample_id].copy().sort_values('date').reset_index(drop=True)
print(f'Sample series: {sample_id}')
print(f'Length: {len(sample)} days')

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(sample['date'], sample['sales'], linewidth=0.7)
ax.set_title(f'Raw Sales: {sample_id}')
plt.tight_layout()
plt.show()

### 2a. Lag Features

In [ ]:
sample = add_lag_features(sample, 'sales', [7, 14, 28], group_col='id')

fig, ax = plt.subplots(figsize=(16, 5))
last_90 = sample.tail(90)
ax.plot(last_90['date'], last_90['sales'], label='actual', linewidth=1.5)
ax.plot(last_90['date'], last_90['lag_7'], label='lag_7', alpha=0.7)
ax.plot(last_90['date'], last_90['lag_28'], label='lag_28', alpha=0.7)
ax.set_title('Sales vs Lagged Values (Last 90 Days)')
ax.legend()
plt.tight_layout()
plt.show()

### 2b. Rolling Statistics

In [ ]:
sample = add_rolling_features(sample, 'sales', [7, 28], ['mean', 'std'], group_col='id')

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
last_180 = sample.tail(180)

axes[0].plot(last_180['date'], last_180['sales'], alpha=0.5, label='actual')
axes[0].plot(last_180['date'], last_180['rolling_mean_7'], label='7d mean', linewidth=2)
axes[0].plot(last_180['date'], last_180['rolling_mean_28'], label='28d mean', linewidth=2)
axes[0].set_title('Rolling Means')
axes[0].legend()

axes[1].plot(last_180['date'], last_180['rolling_std_7'], label='7d std')
axes[1].plot(last_180['date'], last_180['rolling_std_28'], label='28d std')
axes[1].set_title('Rolling Standard Deviations')
axes[1].legend()

plt.tight_layout()
plt.show()

### 2c. Fourier Features

In [ ]:
sample = add_fourier_features(sample, 'date', period=365.25, order=3)

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(sample['date'], sample['fourier_sin_1'], label='sin_1', alpha=0.8)
ax.plot(sample['date'], sample['fourier_cos_1'], label='cos_1', alpha=0.8)
ax.plot(sample['date'], sample['fourier_sin_2'], label='sin_2', alpha=0.5)
ax.set_title('Fourier Terms (Yearly Seasonality)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Full Feature Pipeline

In [ ]:
df_featured = build_features(df, config, include_target_encoding=True)
print(f'\nFinal shape: {df_featured.shape}')
print(f'Features: {df_featured.shape[1]}')
print(f'\nFeature columns:')
for i, col in enumerate(sorted(df_featured.columns)):
    print(f'  {i+1:3d}. {col}')

## 4. Feature Correlation

In [ ]:
numeric_cols = df_featured.select_dtypes(include=[np.number]).columns
target_corr = df_featured[numeric_cols].corrwith(df_featured['sales']).drop('sales').sort_values()

fig, ax = plt.subplots(figsize=(10, max(6, len(target_corr) * 0.2)))
colors = ['salmon' if v < 0 else 'steelblue' for v in target_corr.values]
target_corr.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Correlation with Sales')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Save Featured Dataset

In [ ]:
out_path = '../data/processed/sales_featured.parquet'
df_featured.to_parquet(out_path, index=False)
import os
print(f'Saved: {out_path} ({os.path.getsize(out_path) / 1024**2:.1f} MB)')